In [183]:
import requests
import pandas as pd
import numpy as np

In [ ]:
def get_user_id(name):
    num_results = 1
    try:
        page = requests.get(f'https://app.universaltennis.com/api/v2/search/players?query={name}&top={num_results}')
        data = page.json()
        # extract user id from either 'hits' or 'Hits' because utr has two JSON responses
        key = 'hits' if 'hits' in data else 'Hits'
        return data[key][0]['id'] if key == 'hits' else data[key][0]['Id']
    except (KeyError, IndexError, requests.exceptions.RequestException) as e:
        print(f'Error: Could not find user ID for {name}')

In [182]:
get_user_id('Sebastian Nothaft')

Error Occurred: list index out of range


In [126]:
get_user_id('Roger Federer')

'4001303'

In [4]:
def get_utr(name):
    user_id = get_user_id(name)
    # get player utr
    url = f'https://app.universaltennis.com/api/v1/player/{user_id}'
    page = requests.get(url)
    data = page.json()
    singles_utr = data['singlesUtr']
    return singles_utr

In [5]:
print(get_utr('Novak Djokovic'))
print(get_utr('Spencer Johnson'))

16.23
13.45


In [5]:
ucla_players = ['Gianluca Ballotta',
                'Kaylan Bigun',
                'Cassius Chinlund',
                'Andrei Crabel',
                'Alexander Hoogmartens',
                'Spencer Johnson',
                'Rudy Quan',
                'Giacomo Revelli',
                'Aadarsh Tripathi',
                'Emon van Loben Sels',
                'Leo von Bismarck',
                
                'Olivia Center',
                'Kate Fakih',
                'Bianca Fernandez',
                'Ahmani Guichard',
                'Kimberly Hance',
                'Mia Jovic',
                'Anne-Christine Lutkemeyer',
                'Elise Wagle']

In [6]:
# get UTRs of all ucla players and store in dictionary
ucla_utr_dict = {}
for player in ucla_players:
    print(player)
    utr = get_utr(player)
    ucla_utr_dict[player] = utr

Gianluca Ballotta
Kaylan Bigun
Cassius Chinlund
Andrei Crabel
Alexander Hoogmartens
Spencer Johnson
Rudy Quan
Giacomo Revelli
Aadarsh Tripathi
Emon van Loben Sels
Leo von Bismarck
Olivia Center
Kate Fakih
Bianca Fernandez
Ahmani Guichard
Kimberly Hance
Mia Jovic
Anne-Christine Lutkemeyer
Elise Wagle


In [7]:
ucla_utr_dict

{'Gianluca Ballotta': 12.0,
 'Kaylan Bigun': 13.19,
 'Cassius Chinlund': 12.27,
 'Andrei Crabel': 11.48,
 'Alexander Hoogmartens': 13.0,
 'Spencer Johnson': 13.5,
 'Rudy Quan': 13.91,
 'Giacomo Revelli': 12.0,
 'Aadarsh Tripathi': 12.97,
 'Emon van Loben Sels': 13.31,
 'Leo von Bismarck': 11.0,
 'Olivia Center': 10.26,
 'Kate Fakih': 10.78,
 'Bianca Fernandez': 9.99,
 'Ahmani Guichard': 10.0,
 'Kimberly Hance': 10.43,
 'Mia Jovic': 8.0,
 'Anne-Christine Lutkemeyer': 11.0,
 'Elise Wagle': 10.0}

## Getting Dataframe of Past Matches

#### getting singles results

player 1 is winner

In [6]:
# Convert UTR scoring into a string
def get_score_string(score):
    sets = []
    for i in range(1, 4):
        if str(i) in score:
            loser_score = score[str(i)]['loser'] if score[str(i)]['loser'] is not None else '0' # account for 0 stored as None in the UTR data
            set_score = f"{score[str(i)]['winner']}-{loser_score}"
            if score[str(i)]['tiebreak'] != None:
                set_score += f"({score[str(i)]['tiebreak']})"
            sets.append(set_score)
    return ', '.join(sets)

In [7]:
from datetime import datetime
from zoneinfo import ZoneInfo

In [10]:
# Convert date into a more readable format
def convert_date(date):
    utc = datetime.fromisoformat(date).replace(tzinfo=ZoneInfo('UTC'))
    pst = utc.astimezone(ZoneInfo('America/Los_Angeles'))
    return pst.strftime('%Y-%m-%d')

In [11]:
def get_singles_results(name):
    user_id = get_user_id(name)

    # add ?type=singles for singles only
    url = f'https://app.universaltennis.com/api/v1/player/{user_id}/results?type=singles'
    page = requests.get(url)
    data = page.json()

    results = []
    for event in data['events']:
        event_name = event['name']

        if len(event['draws']) == 0:
            for result in event['results']:
                date = convert_date(result['date'])
                player1 = result['players']['winner1']['firstName'] + ' ' + result['players']['winner1']['lastName']
                player2 = result['players']['loser1']['firstName'] + ' ' + result['players']['loser1']['lastName']
                player1_singles_utr = result['players']['winner1']['singlesUtr']
                player2_singles_utr = result['players']['loser1']['singlesUtr']
                score = get_score_string(result['score'])
                results.append([event_name, date, player1, player1_singles_utr, player2, player2_singles_utr, score])
        else:
            for result in event['draws'][0]['results']:
                date = convert_date(result['date'])
                player1 = result['players']['winner1']['firstName'] + ' ' + result['players']['winner1']['lastName']
                player2 = result['players']['loser1']['firstName'] + ' ' + result['players']['loser1']['lastName']
                player1_singles_utr = result['players']['winner1']['singlesUtr']
                player2_singles_utr = result['players']['loser1']['singlesUtr']
                score = get_score_string(result['score'])
                results.append([event_name, date, player1, player1_singles_utr, player2, player2_singles_utr, score])

    singles_results = pd.DataFrame(results, columns=['Event Name', 'Date', 'Player1', 'Player1 UTR', 'Player2', 'Player2 UTR', 'Score'])
    return singles_results

In [167]:
pd.set_option('display.max_rows', None)

In [168]:
singles = get_singles_results('Spencer Johnson')
singles

,Event Name,Date,Player1,Player1 UTR,Player2,Player2 UTR,Score
0,"Dual Match: University of California, Los Ange...",2025-05-02,Spencer Johnson,13.45,Derrick Chen,13.00,"6-2, 6-3"
1,"Dual Match: University of California, Santa Ba...",2025-05-01,Spencer Johnson,13.45,Charlie Underwood,12.00,"7-6(3), 6-2"
2,"Dual Match: University of California, Los Ange...",2025-04-26,Bryce Nakashima Nakashima,13.34,Spencer Johnson,13.45,"6-2, 6-4"
3,Dual Match: Michigan State University vs Unive...,2025-04-25,Spencer Johnson,13.45,Mitchell Sheldon,12.61,"6-2, 6-3"
4,Dual Match: University of Michigan vs Universi...,2025-04-24,Spencer Johnson,13.45,Alex Cairo,12.33,"6-1, 6-1"
5,"Dual Match: University of California, Los Ange...",2025-04-19,Roni Hietaranta,12.00,Spencer Johnson,13.45,"3-6, 6-4, 6-6(4)"
6,"Dual Match: University of California, Los Ange...",2025-04-17,Matthew Fullerton,12.00,Spencer Johnson,13.45,"6-1, 5-5"
7,Dual Match: Stanford University vs University ...,2025-02-01,Samir Banerjee,13.68,Spencer Johnson,13.45,"6-3, 3-6, 6-4"
8,"Dual Match: University of California, Berkeley...",2025-01-31,Spencer Johnson,13.45,Alexander Chang,13.00,"7-6(1), 6-7(7), 2-2"
9,Dual Match: University of Georgia vs Universit...,2025-01-25,Spencer Johnson,13.45,Thomas Paulsell,13.00,"6-2, 7-6(3)"


## Trying to get historical UTR

In [21]:
import pprint

In [13]:
get_user_id('Spencer Johnson')

'2580878'

In [83]:
# Spencer Johnson statistics
url = f'https://app.universaltennis.com/api/v1/player/2580878/stats?type=singles&resultType=verified&Months=12&fetchAllResults=false'
page = requests.get(url)
spencer_stats = page.json()

In [84]:
pprint.pp(spencer_stats)

{'title': 'verified singles stats - Last 12 Months',
 'subtitle': 'June 2024 - May 2025',
 'monthsCount': 12,
 'currentRating': 13.46,
 'currentRatingDisplay': '13.45',
 'ratingCircle': {'singlesUtr': 13.45,
                  'ratingStatusSingles': 'Rated',
                  'ratingProgressSingles': 100.0,
                  'singlesUtrDisplay': '13.45'},
 'winsCount': 19,
 'winStreak': 9,
 'lossesCount': 8,
 'recordWinPercentage': '70.4%',
 'notableMatch': {'winner': {'isWinner': True,
                             'set1': 6,
                             'set2': 6,
                             'set3': 0,
                             'set4': 0,
                             'set5': 0,
                             'set6': None,
                             'tiebreakerSet1': None,
                             'tiebreakerSet2': None,
                             'tiebreakerSet3': 0,
                             'tiebreakerSet4': 0,
                             'tiebreakerSet5': 0,
          

In [85]:
# UTR history for each week
spencer_stats['extendedRatingProfile']['history']

[{'id': 510788615,
  'ratingStatus': 3,
  'rating': 13.47,
  'ratingDisplay': '13.47',
  'type': 'WeeklyAverage_Singles',
  'date': '2025-05-05T00:00:00'},
 {'id': 509769060,
  'ratingStatus': 3,
  'rating': 13.52,
  'ratingDisplay': '13.52',
  'type': 'WeeklyAverage_Singles',
  'date': '2025-04-28T00:00:00'},
 {'id': 508210034,
  'ratingStatus': 3,
  'rating': 13.52,
  'ratingDisplay': '13.52',
  'type': 'WeeklyAverage_Singles',
  'date': '2025-04-21T00:00:00'},
 {'id': 507103189,
  'ratingStatus': 3,
  'rating': 13.51,
  'ratingDisplay': '13.51',
  'type': 'WeeklyAverage_Singles',
  'date': '2025-04-14T00:00:00'},
 {'id': 505639210,
  'ratingStatus': 3,
  'rating': 13.51,
  'ratingDisplay': '13.51',
  'type': 'WeeklyAverage_Singles',
  'date': '2025-04-07T00:00:00'},
 {'id': 504661842,
  'ratingStatus': 3,
  'rating': 13.51,
  'ratingDisplay': '13.51',
  'type': 'WeeklyAverage_Singles',
  'date': '2025-03-31T00:00:00'},
 {'id': 503864848,
  'ratingStatus': 3,
  'rating': 13.51,
  'ra

In [38]:
spencer_stats['extendedRatingProfile']['history'][0]

{'id': 510788615,
 'ratingStatus': 3,
 'rating': 13.47,
 'ratingDisplay': '13.47',
 'type': 'WeeklyAverage_Singles',
 'date': '2025-05-05T00:00:00'}

In [49]:
def create_UTR_date_dict(utr_history):
    utr_date_dict = {}
    for week in utr_history:
        date = datetime.strptime(week['date'], '%Y-%m-%dT%H:%M:%S').date()
        rating = week['ratingDisplay']
        utr_date_dict[date] = rating
    return utr_date_dict

In [82]:
utr_dict = create_UTR_date_dict(spencer_stats['extendedRatingProfile']['history'])
utr_dict

{datetime.date(2025, 5, 5): '13.47',
 datetime.date(2025, 4, 28): '13.52',
 datetime.date(2025, 4, 21): '13.52',
 datetime.date(2025, 4, 14): '13.51',
 datetime.date(2025, 4, 7): '13.51',
 datetime.date(2025, 3, 31): '13.51',
 datetime.date(2025, 3, 24): '13.51',
 datetime.date(2025, 3, 17): '13.51',
 datetime.date(2025, 3, 10): '13.51',
 datetime.date(2025, 3, 3): '13.51',
 datetime.date(2025, 2, 24): '13.51',
 datetime.date(2025, 2, 17): '13.51',
 datetime.date(2025, 2, 10): '13.51',
 datetime.date(2025, 2, 3): '13.52',
 datetime.date(2025, 1, 27): '13.59',
 datetime.date(2025, 1, 20): '13.59',
 datetime.date(2025, 1, 13): '13.59',
 datetime.date(2025, 1, 6): '13.56',
 datetime.date(2024, 12, 30): '13.54',
 datetime.date(2024, 12, 23): '13.52',
 datetime.date(2024, 12, 16): '13.49',
 datetime.date(2024, 12, 9): '13.45',
 datetime.date(2024, 12, 2): '13.41',
 datetime.date(2024, 11, 25): '13.37',
 datetime.date(2024, 11, 18): '13.36',
 datetime.date(2024, 11, 11): '13.36',
 datetime.d

In [ ]:
# get historical UTR at time of match, for a specific player
# match the event date to the closest date in the UTR history
def get_historical_utr(name, date):
    user_id = get_user_id(name)
    
    url = f'https://app.universaltennis.com/api/v1/player/{user_id}/stats?type=singles&resultType=verified&Months=12&fetchAllResults=false'
    page = requests.get(url)
    stats = page.json()

    utr_history = stats['extendedRatingProfile']['history']
    utr_dict = create_UTR_date_dict(utr_history)

In [76]:
def get_closest_date(utr_dict, date):
    closest_date = None
    closest_diff = None
    for key in utr_dict.keys():
        diff = abs((key - date).days)
        if closest_diff is None or diff < closest_diff:
            closest_diff = diff
            closest_date = key
    return closest_date

In [81]:
closest = get_closest_date(utr_dict, datetime.strptime('2024-10-01', '%Y-%m-%d').date())
closest

datetime.date(2024, 9, 30)

In [80]:
utr_dict[closest]

'13.40'

bro idk

In [202]:
name = 'Spencer Johnson'
user_id = get_user_id(name)

url_stats = f'https://app.universaltennis.com/api/v1/player/{user_id}/stats?type=singles&resultType=verified&Months=12&fetchAllResults=false'
page = requests.get(url_stats)
stats = page.json()

url_singles = f'https://app.universaltennis.com/api/v1/player/{user_id}/results?type=singles'
page = requests.get(url_singles)
singles = page.json()

In [203]:
pprint.pp(stats)

{'title': 'verified singles stats - Last 12 Months',
 'subtitle': 'June 2024 - May 2025',
 'monthsCount': 12,
 'currentRating': 13.46,
 'currentRatingDisplay': '13.45',
 'ratingCircle': {'singlesUtr': 13.45,
                  'ratingStatusSingles': 'Rated',
                  'ratingProgressSingles': 100.0,
                  'singlesUtrDisplay': '13.45'},
 'winsCount': 19,
 'winStreak': 9,
 'lossesCount': 8,
 'recordWinPercentage': '70.4%',
 'notableMatch': {'winner': {'isWinner': True,
                             'set1': 6,
                             'set2': 6,
                             'set3': 0,
                             'set4': 0,
                             'set5': 0,
                             'set6': None,
                             'tiebreakerSet1': None,
                             'tiebreakerSet2': None,
                             'tiebreakerSet3': 0,
                             'tiebreakerSet4': 0,
                             'tiebreakerSet5': 0,
          

In [152]:
stats['extendedRatingProfile'] == None

True

In [161]:
import unicodedata
def remove_accents(string):
    nfkd_form = unicodedata.normalize('NFKD', string)
    return u"".join([c for c in nfkd_form if not unicodedata.combining(c)])

In [166]:
remove_accents('Théo PAPAMALAMIS')

'Theo PAPAMALAMIS'

In [171]:
dict = {
    'firstName': 'Nicolas '
                'Ian',
    'lastName': 'Kotzen',
}

In [176]:
dict['firstName'].split()[0]

'Nicolas'

In [ ]:
# if player1 == 'Spencer Johnson' then assign player1 historical utr to the utr

In [187]:
def get_historical_UTR(name, match_date):
    print(name)
    # create dicitonary of (weekly) dates and UTR ratings
    user_id = get_user_id(name)
    url = f'https://app.universaltennis.com/api/v1/player/{user_id}/stats?type=singles&resultType=verified&Months=12&fetchAllResults=false'
    page = requests.get(url)
    stats = page.json()

    # for now, if there's no extendedRatingProfile history, return nan
    try:
        utr_history = stats['extendedRatingProfile']['history']
    except (KeyError, IndexError, TypeError):
        print(f'Error: Could not find UTR history for {name}')
        return np.nan

    utr_dict = {}
    for week in utr_history:
        date = datetime.strptime(week['date'], '%Y-%m-%dT%H:%M:%S').date()
        rating = week['ratingDisplay']
        utr_dict[date] = rating

    # get the UTR rating from the closest date
    closest_date = None
    closest_diff = None
    for key in utr_dict.keys():
        diff = abs((key - match_date).days)
        if closest_diff is None or diff < closest_diff:
            closest_diff = diff
            closest_date = key

    return utr_dict[closest_date]

In [189]:
name = 'Alex Cairo'
date = datetime.strptime('2025-04-25', '%Y-%m-%d').date()
user_id = get_user_id(name)
url = f'https://app.universaltennis.com/api/v1/player/{user_id}/stats?type=singles&resultType=verified&Months=12&fetchAllResults=false'
page = requests.get(url)
stats = page.json()

In [190]:
try:
    utr_history = stats['extendedRatingProfile']['history']
except (KeyError, IndexError, TypeError):
    print(f'Error: Could not find UTR history for {name}')

In [197]:
utr_history == None

True

In [188]:
get_historical_UTR('Alex Cairo', datetime.strptime('2025-04-25', '%Y-%m-%d').date())

Alex Cairo


TypeError: 'NoneType' object is not iterable